In [1]:
from google.colab import files

uploaded = files.upload()

Saving ED_Simulated_500_Patient_Dataset.csv to ED_Simulated_500_Patient_Dataset.csv


In [2]:
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output

In [3]:
# Load the existing 500-patient dataset

file_name = "ED_Simulated_500_Patient_Dataset.csv"

df = pd.read_csv(file_name)

print("Emergency Department Dataset Loaded")
print("------------------------------------")
print("Number of patients :", len(df))
print("Number of variables:", len(df.columns))

print("\nColumn names:")
for column in df.columns:
    print("-", column)

Emergency Department Dataset Loaded
------------------------------------
Number of patients : 500
Number of variables: 15

Column names:
- Patient_ID
- Arrival_Date
- Arrival_Time
- Priority
- Triage_Start
- Triage_End
- Triage_Wait_Min
- Assessment_Start
- Assessment_Duration_Min
- Treatment_Start
- Treatment_Duration_Min
- Doctor
- Nurse
- Outcome
- Total_Time_in_ED_Min


In [4]:
# Convert important date/time fields

df["Arrival_DateTime"] = pd.to_datetime(
    df["Arrival_Date"].astype(str) + " " +
    df["Arrival_Time"].astype(str)
)

df["Triage_Start"] = pd.to_datetime(
    df["Triage_Start"]
)

df["Triage_End"] = pd.to_datetime(
    df["Triage_End"]
)

df["Assessment_Start"] = pd.to_datetime(
    df["Assessment_Start"]
)

df["Treatment_Start"] = pd.to_datetime(
    df["Treatment_Start"]
)

# Sort patients by arrival
df = df.sort_values(
    "Arrival_DateTime"
).reset_index(drop=True)

print("Data preparation completed.")

Data preparation completed.


In [5]:
def calculate_model_performance(data):

    total_patients = len(data)

    # --------------------------------------------------
    # WAITING TIME
    # --------------------------------------------------

    average_waiting = data[
        "Triage_Wait_Min"
    ].mean()

    maximum_waiting = data[
        "Triage_Wait_Min"
    ].max()

    # --------------------------------------------------
    # SERVICE TIMES
    # --------------------------------------------------

    average_assessment = data[
        "Assessment_Duration_Min"
    ].mean()

    average_treatment = data[
        "Treatment_Duration_Min"
    ].mean()

    # --------------------------------------------------
    # TOTAL ED TIME
    # --------------------------------------------------

    average_ed_time = data[
        "Total_Time_in_ED_Min"
    ].mean()

    maximum_ed_time = data[
        "Total_Time_in_ED_Min"
    ].max()

    # --------------------------------------------------
    # THROUGHPUT
    # --------------------------------------------------

    start_time = data[
        "Arrival_DateTime"
    ].min()

    end_time = data[
        "Arrival_DateTime"
    ].max()

    simulation_hours = (
        end_time - start_time
    ).total_seconds() / 3600

    if simulation_hours > 0:

        throughput = (
            total_patients /
            simulation_hours
        )

    else:

        throughput = 0

    # --------------------------------------------------
    # PATIENT PRIORITY
    # --------------------------------------------------

    priority_distribution = (
        data["Priority"]
        .value_counts()
    )

    # --------------------------------------------------
    # PATIENT OUTCOME
    # --------------------------------------------------

    outcome_distribution = (
        data["Outcome"]
        .value_counts()
    )

    # --------------------------------------------------
    # DOCTOR WORKLOAD
    # --------------------------------------------------

    doctor_workload = (
        data["Doctor"]
        .value_counts()
        .sort_index()
    )

    # --------------------------------------------------
    # NURSE WORKLOAD
    # --------------------------------------------------

    nurse_workload = (
        data["Nurse"]
        .value_counts()
        .sort_index()
    )

    # --------------------------------------------------
    # PERFORMANCE RESULTS
    # --------------------------------------------------

    performance = {

        "Total Patients":
            total_patients,

        "Average Waiting Time (min)":
            round(
                average_waiting,
                2
            ),

        "Maximum Waiting Time (min)":
            round(
                maximum_waiting,
                2
            ),

        "Average Assessment Time (min)":
            round(
                average_assessment,
                2
            ),

        "Average Treatment Time (min)":
            round(
                average_treatment,
                2
            ),

        "Average Total ED Time (min)":
            round(
                average_ed_time,
                2
            ),

        "Maximum Total ED Time (min)":
            round(
                maximum_ed_time,
                2
            ),

        "Throughput (patients/hour)":
            round(
                throughput,
                2
            )
    }

    return (
        performance,
        priority_distribution,
        outcome_distribution,
        doctor_workload,
        nurse_workload
    )

In [6]:
(
    performance,
    priority_distribution,
    outcome_distribution,
    doctor_workload,
    nurse_workload
) = calculate_model_performance(df)

print("=" * 60)
print("HOSPITAL EMERGENCY DEPARTMENT")
print("PERFORMANCE MODEL RESULTS")
print("=" * 60)

for measure, value in performance.items():

    print(
        f"{measure}: {value}"
    )

HOSPITAL EMERGENCY DEPARTMENT
PERFORMANCE MODEL RESULTS
Total Patients: 500
Average Waiting Time (min): 0.1
Maximum Waiting Time (min): 13
Average Assessment Time (min): 31.09
Average Treatment Time (min): 60.25
Average Total ED Time (min): 99.82
Maximum Total ED Time (min): 195
Throughput (patients/hour): 0.68


In [7]:
print("=" * 60)
print("DOCTOR WORKLOAD")
print("=" * 60)

doctor_table = doctor_workload.to_frame(
    name="Patients Assigned"
)

display(doctor_table)

print("\n")
print("=" * 60)
print("NURSE WORKLOAD")
print("=" * 60)

nurse_table = nurse_workload.to_frame(
    name="Patients Assigned"
)

display(nurse_table)

DOCTOR WORKLOAD


,Patients Assigned
Doctor,
Doctor_1,127
Doctor_2,122
Doctor_3,123
Doctor_4,128




NURSE WORKLOAD


,Patients Assigned
Nurse,
Nurse_1,78
Nurse_2,92
Nurse_3,81
Nurse_4,93
Nurse_5,92
Nurse_6,64


In [8]:
print("=" * 60)
print("PATIENT PRIORITY DISTRIBUTION")
print("=" * 60)

display(
    priority_distribution.to_frame(
        name="Number of Patients"
    )
)

print("\n")
print("=" * 60)
print("PATIENT OUTCOME DISTRIBUTION")
print("=" * 60)

display(
    outcome_distribution.to_frame(
        name="Number of Patients"
    )
)

PATIENT PRIORITY DISTRIBUTION


,Number of Patients
Priority,
Non-Urgent,244
Urgent,202
Critical,54




PATIENT OUTCOME DISTRIBUTION


,Number of Patients
Outcome,
Discharged,399
Admitted,101


In [9]:
# ==========================================================
# HOSPITAL EMERGENCY DEPARTMENT
# PERFORMANCE MODELLING GUI
# ==========================================================

title = widgets.HTML(
    value="""
    <h2>🏥 Hospital Emergency Department</h2>
    <h3>Performance Modelling and Simulation</h3>
    """
)

patient_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=500,
    step=100,
    description="Patients:",
    style={
        "description_width": "initial"
    }
)

doctor_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=8,
    step=1,
    description="Doctors:",
    style={
        "description_width": "initial"
    }
)

nurse_slider = widgets.IntSlider(
    value=6,
    min=1,
    max=10,
    step=1,
    description="Nurses:",
    style={
        "description_width": "initial"
    }
)

scenario_dropdown = widgets.Dropdown(
    options=[
        "Normal Load",
        "High Patient Load",
        "Increased Staff"
    ],
    value="Normal Load",
    description="Scenario:",
    style={
        "description_width": "initial"
    }
)

run_button = widgets.Button(
    description="RUN MODEL",
    button_style="success"
)

gui_output = widgets.Output()

In [10]:
def run_gui_model(button):

    with gui_output:

        clear_output(wait=True)

        # ----------------------------------------------
        # SELECT NUMBER OF PATIENTS
        # ----------------------------------------------

        number_of_patients = (
            patient_slider.value
        )

        model_data = df.head(
            number_of_patients
        ).copy()

        # ----------------------------------------------
        # RESOURCE SETTINGS
        # ----------------------------------------------

        doctors = doctor_slider.value
        nurses = nurse_slider.value

        scenario = (
            scenario_dropdown.value
        )

        # ----------------------------------------------
        # PERFORMANCE MEASURES
        # ----------------------------------------------

        average_wait = (
            model_data[
                "Triage_Wait_Min"
            ].mean()
        )

        maximum_wait = (
            model_data[
                "Triage_Wait_Min"
            ].max()
        )

        average_assessment = (
            model_data[
                "Assessment_Duration_Min"
            ].mean()
        )

        average_treatment = (
            model_data[
                "Treatment_Duration_Min"
            ].mean()
        )

        average_ed_time = (
            model_data[
                "Total_Time_in_ED_Min"
            ].mean()
        )

        # ----------------------------------------------
        # WORKLOAD
        # ----------------------------------------------

        doctor_assignments = (
            model_data[
                "Doctor"
            ].value_counts()
        )

        nurse_assignments = (
            model_data[
                "Nurse"
            ].value_counts()
        )

        total_doctor_work = (
            doctor_assignments.sum()
        )

        total_nurse_work = (
            nurse_assignments.sum()
        )

        doctor_workload = (
            total_doctor_work /
            doctors
        )

        nurse_workload = (
            total_nurse_work /
            nurses
        )

        # ----------------------------------------------
        # DISPLAY GUI RESULTS
        # ----------------------------------------------

        print("=" * 65)
        print("🏥 HOSPITAL EMERGENCY DEPARTMENT")
        print("PERFORMANCE MODELLING SYSTEM")
        print("=" * 65)

        print(
            f"\nScenario: {scenario}"
        )

        print(
            f"Patients considered: "
            f"{number_of_patients}"
        )

        print(
            f"Doctors available: "
            f"{doctors}"
        )

        print(
            f"Nurses available: "
            f"{nurses}"
        )

        print("\nPERFORMANCE MEASURES")
        print("-" * 65)

        print(
            f"Average waiting time: "
            f"{average_wait:.2f} minutes"
        )

        print(
            f"Maximum waiting time: "
            f"{maximum_wait:.2f} minutes"
        )

        print(
            f"Average assessment time: "
            f"{average_assessment:.2f} minutes"
        )

        print(
            f"Average treatment time: "
            f"{average_treatment:.2f} minutes"
        )

        print(
            f"Average total ED time: "
            f"{average_ed_time:.2f} minutes"
        )

        print("\nRESOURCE WORKLOAD")
        print("-" * 65)

        print(
            f"Doctor workload indicator: "
            f"{doctor_workload:.2f} patients/doctor"
        )

        print(
            f"Nurse workload indicator: "
            f"{nurse_workload:.2f} patients/nurse"
        )

        print("\nPATIENT OUTCOMES")
        print("-" * 65)

        display(
            model_data[
                "Outcome"
            ].value_counts().to_frame(
                name="Patients"
            )
        )


run_button.on_click(
    run_gui_model
)

display(
    widgets.VBox([
        title,
        patient_slider,
        doctor_slider,
        nurse_slider,
        scenario_dropdown,
        run_button,
        gui_output
    ])
)